# 12. Visualization Test

Pipeline step ⑫ checks whether pose coordinates can be visualized as a 3D skeleton animation for reporting/QC.

Previous checks are assumed to be working:
- setup_00_environment_check
- setup_01_data_loading_test
- 01_validation_test


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

from movement.io import load_pose_csv
from movement.config import LANDMARKS, CONNECTIONS
from movement.visualization import create_pose_animation

df = load_pose_csv(PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv")

def estimate_frame_duration_ms(dataframe, default_ms=33):
    if "timestamp" not in dataframe.columns:
        return default_ms
    dt = dataframe["timestamp"].astype(float).diff().dropna()
    if dt.empty:
        return default_ms
    median_dt = float(dt.median())
    if median_dt <= 0:
        return default_ms
    return max(1, int(round(median_dt * 1000)))

frame_duration_ms = estimate_frame_duration_ms(df)
print(f"playback frame duration: {frame_duration_ms} ms (~{1000 / frame_duration_ms:.1f} fps)")

fig = create_pose_animation(
    df=df,
    landmarks=LANDMARKS,
    connections=CONNECTIONS,
    coord_mode="raw",
    title="Visualization Stage - Recording View Pose",
    show_text=False,
    frame_duration=frame_duration_ms,
)

recording_view_camera = dict(
    eye=dict(x=0.0, y=-2.5, z=0.0),
    center=dict(x=0.0, y=0.0, z=0.0),
    up=dict(x=0.0, y=0.0, z=1.0),
    projection=dict(type="orthographic"),
)
fig.update_layout(scene_camera=recording_view_camera)

fig.show()
